# All sites plot Land Exposure compared to Low Tide

For the selected date range, process Sentinel-2 scenes with less than 10% catalogue cloud cover at all configured sites. For each scene, record the nearest predicted low-tide time, height, and signed time offset; estimate whole-site dry area from SCL vegetation (class 4) and not-vegetated (class 5) pixels. Save per-site CSVs, RGB/SCL galleries, and dry-area-versus-low-tide scatter plots in each site's output folder.

In [1]:
import base64
import datetime
import html as html_module
import io
import os
import sys
import zoneinfo
import pathlib

import dotenv
import geopandas
import matplotlib.pyplot as plt
import numpy
import odc.stac
import pandas
import planetary_computer
import plotly.express as px
import rasterio
import rasterio.features
import requests
import xarray
from IPython.display import FileLink, display

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

In [2]:
data_path = utils.get_data_path()
site_polygons_path = data_path / "site_polygons"
output_root = data_path / "analysis" / "all_sites"
output_root.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv()
tide_api_key = os.environ.get("TIDE_API")
if not tide_api_key:
    raise ValueError("Set TIDE_API in the environment or project .env file before querying tides.")

## Date-Range Site and Cloud Settings

Set the start and end years and cloud-cover cutoff below. All matching scenes are retained; low-tide timing is recorded for comparison, not used to exclude scenes.

In [7]:
START_YEAR = 2016
END_YEAR = 2025
DATE_RANGE_LABEL = f"{START_YEAR}-{END_YEAR}"
MAX_ITEMS = 400
MAX_CLOUD_COVER_PCT = 10.0
TIDE_SEARCH_RADIUS_HRS = 12

print(f"Years: {DATE_RANGE_LABEL}; STAC item cap per site per year: {MAX_ITEMS}")
print(f"Scene cloud-cover requirement: <{MAX_CLOUD_COVER_PCT:g}%")
print("Dry area: SCL 4 + 5 pixels divided by all rasterized site pixels")

Years: 2016-2025; STAC item cap per site per year: 400
Scene cloud-cover requirement: <10%
Dry area: SCL 4 + 5 pixels divided by all rasterized site pixels


In [8]:
SITES = ("Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24")
SITE_POLYGON_FILES = {
    site_name: site_polygons_path / f"{site_name}_polygon.gpkg"
    for site_name in SITES
}
SITE_OUTPUT_PATHS = {
    site_name: output_root / site_name
    for site_name in SITES
}

for site_name, polygon_file in SITE_POLYGON_FILES.items():
    if not polygon_file.is_file():
        raise FileNotFoundError(f"Missing site polygon for {site_name}: {polygon_file}")
    SITE_OUTPUT_PATHS[site_name].mkdir(parents=True, exist_ok=True)

print(f"Sites: {', '.join(SITES)}")
print(f"Per-site output root: {output_root}")

Sites: Ihutai, CatlinsLake, CatlinsRiverMouth, Childrens, Duvauchelle, Robinsons, Takamatua, Purau, IveyBay_Nov25, IveyBay_Feb26, LeftBank_Nov25, LeftBank_Feb26, Paremata_Nov25, Paremata_Feb26, Paremata_Feb25, ThePoint_Nov25, ThePoint_Feb26, Takapuwahia_Nov25, Takapuwahia_Feb26, IveyBay_ThePoint_LeftBank_Oct24
Per-site output root: C:\Local\repos\seagrass-detection\data\analysis\all_sites


In [21]:
def get_low_tide_info(satellite_datetime_utc, lat, lon):
    """Returns the low tide information for the given satellite image and
      location - low tide height, time and time between the low tide and
      satellite image acquisition.
      lat: Latitude of the location in NZGD49
      lon: Longitude of the location in NZGD49
      """

    dotenv.load_dotenv()
    tide_api_key = os.environ.get("TIDE_API", None)
    if tide_api_key is None:
        raise ValueError("TIDE_API environment variable not set in .env file")

    satellite_time_nz_dt = satellite_datetime_utc.astimezone(
        zoneinfo.ZoneInfo('Pacific/Auckland'))
    start_date = ( # get the date 7hrs before the NZDT satellite time
        satellite_time_nz_dt - datetime.timedelta(hours=7)
        ).strftime(sentinel2.DATE_FORMAT_YYYYMMDD)
    tide_url = ( # Get tide for at least one low tide cyle either side
        f"{sentinel2.TIDE_API_STUB}?lat={lat}&long={lon}&datum=MSL"
        f"&numberOfDays=2&apikey={tide_api_key}&startDate={start_date}"
    )

    tide_query = requests.get(tide_url)
    tide_query.raise_for_status()
    
    tides = pandas.DataFrame.from_records(tide_query.json()["values"])
    tides["time"] = pandas.to_datetime(tides["time"], utc=True)
    tides = tides[tides["value"] < 0] # Low tides only

    time_from_low_tide = (tides["time"] - satellite_datetime_utc).abs().min()
    nearest_low = tides.loc[(tides["time"] - satellite_datetime_utc).abs().idxmin()]

    tide_info = { 
        "nearest_low_tide_height_msl": nearest_low["value"],
        "nearest_low_tide_time_utc": nearest_low["time"],
        "time_from_low_tide": format_tide_duration(time_from_low_tide),
        "hours_from_low_tide": time_from_low_tide.total_seconds() / 3600,
            }

    return tide_info

In [22]:
def format_tide_duration(time_from_low_tide):
    hours_from_low = time_from_low_tide.total_seconds() / 3600
    total_minutes = int(round(abs(float(hours_from_low)) * 60))
    hours, minutes = divmod(total_minutes, 60)
    direction = "at" if total_minutes == 0 else "before" if hours_from_low < 0 else "after"
    return f"{hours}h {minutes:02d}m {direction}"

In [19]:
scl_classes = [
    (0, "No data", "#000000"),
    (1, "Saturated/defective", "#ff0000"),
    (2, "Dark area", "#2f2f2f"),
    (3, "Cloud shadow", "#643200"),
    (4, "Vegetation", "#00a000"),
    (5, "Not vegetated", "#ffe65a"),
    (6, "Water", "#0000ff"),
    (7, "Unclassified", "#808080"),
    (8, "Cloud, medium probability", "#c0c0c0"),
    (9, "Cloud, high probability", "#ffffff"),
    (10, "Thin cirrus", "#64c8ff"),
    (11, "Snow/ice", "#ff96ff"),
]
scl_legend_html = '<div class="legend">' + "".join(
    f'<span class="legend-item"><i style="background:{color}"></i>{code}: '
    f'{html_module.escape(label)}</span>'
    for code, label, color in scl_classes
) + "</div>"


def make_gallery(site_name, gallery_title, cards, output_file, legend_html=""):
    gallery_style = """
:root { color-scheme: light; font-family: "Segoe UI", sans-serif; color: #202a33; background: #f3f5f6; }
body { margin: 0; padding: 20px; }
header, .gallery, .legend { max-width: 1500px; margin: 0 auto 16px; }
h1 { margin: 0 0 6px; font-size: 22px; }
p { margin: 0; color: #56616b; }
.gallery { display: grid; grid-template-columns: repeat(auto-fill, minmax(min(320px, 100%), 1fr)); gap: 14px; }
.scene { min-width: 0; overflow: hidden; background: white; border: 1px solid #d5dce1; border-radius: 5px; }
.scene h2 { margin: 10px 12px 0; font-size: 15px; }
.scene img { display: block; width: 100%; aspect-ratio: 4 / 3; object-fit: contain; background: #f7f8f8; }
.caption { padding: 8px 12px 11px; font-size: 12px; line-height: 1.4; }
.legend { display: flex; flex-wrap: wrap; gap: 6px 12px; padding: 10px; background: white; border: 1px solid #d5dce1; }
.legend-item { display: inline-flex; align-items: center; gap: 5px; font-size: 11px; }
.legend-item i { display: inline-block; width: 14px; height: 14px; border: 1px solid #7b858c; }
@media (max-width: 600px) { body { padding: 10px; } }
"""
    page = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{html_module.escape(site_name)} {DATE_RANGE_LABEL} {html_module.escape(gallery_title)}</title>
<style>{gallery_style}</style>
</head>
<body>
<header><h1>{html_module.escape(site_name)}: {DATE_RANGE_LABEL} {html_module.escape(gallery_title)}</h1>
<p>{len(cards)} scenes with catalogue cloud cover under {MAX_CLOUD_COVER_PCT:g}%.</p></header>
{legend_html}
<main class="gallery">{''.join(cards)}</main>
</body>
</html>"""
    output_file.write_text(page, encoding="utf-8")
    print(f"  Saved {output_file}")

In [16]:
def harmonize_post_2022(data):
    """Bring post-baseline reflectance bands onto the pre-2022 scale."""

    available_bands = [band for band in sentinel2.BANDS if band in data.data_vars]
    
    post_baseline = data["time"] > numpy.datetime64(sentinel2.HARMONIZE_DATE)
    for band in available_bands:
        if band == "SCL":
            continue
        reflectance = data[band]
        data[band] = xarray.where(
            post_baseline & reflectance.notnull() & (reflectance != 0),
            reflectance.clip(min=sentinel2.BAND_OFFSET_POST_2022_01_25) - sentinel2.BAND_OFFSET_POST_2022_01_25,
            reflectance,
        )
    return data

In [62]:
site_results = {}
for site_name in SITES:
    print(f"\n{site_name}: search {DATE_RANGE_LABEL} Sentinel-2 scenes")
    site_polygon_file = SITE_POLYGON_FILES[site_name]
    site_output_path = SITE_OUTPUT_PATHS[site_name]
    site_polygon = geopandas.read_file(site_polygon_file)
    if site_polygon.crs is None or site_polygon.empty:
        raise ValueError(f"Site polygon must be non-empty and have a CRS: {site_polygon_file}")

    geometry_wgs = site_polygon.buffer(sentinel2.S2_RESOLUTION).to_crs(utils.CRS_WSG)
    stac_items = []
    for year in range(START_YEAR, END_YEAR + 1):
        year_results = sentinel2.stac_search_with_retry(
            url=sentinel2.CATALOGUE_URL,
            max_items=MAX_ITEMS,
            collections=[sentinel2.COLLECTION],
            bbox=geometry_wgs.total_bounds,
            datetime=f"{year}-01-01/{year}-12-31",
            sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}],
            get_collection=True,
        )
        stac_items.extend(year_results.items)
        print(f"  {year}: {len(year_results.items)} STAC items")

    cloud_filtered_items = []
    cloud_cover_by_date = {}
    for item in stac_items:
        cloud_cover = item.properties.get("eo:cloud_cover")
        if cloud_cover is None or float(cloud_cover) >= MAX_CLOUD_COVER_PCT:
            continue
        acquisition_time = pandas.to_datetime(item.properties["datetime"], utc=True)
        scene_date = acquisition_time.strftime("%Y-%m-%d")
        cloud_cover = float(cloud_cover)
        cloud_filtered_items.append(item)
        cloud_cover_by_date[scene_date] = min(
            cloud_cover, cloud_cover_by_date.get(scene_date, cloud_cover)
        )

    print(
        f"  STAC items across range: {len(stac_items)}; "
        f"under cloud cutoff: {len(cloud_filtered_items)}"
    )
    if not cloud_filtered_items:
        print(f"  No scenes below {MAX_CLOUD_COVER_PCT:g}% cloud cover")
        continue

    satellite_data = odc.stac.load(
        cloud_filtered_items,
        bbox=geometry_wgs.total_bounds,
        bands=sentinel2.RGB_BANDS + ["SCL", "B11"],
        chunks={},
        groupby="solar_day",
        resolution=sentinel2.S2_RESOLUTION,
        dtype="uint16",
        nodata=0,
        patch_url=planetary_computer.sign,
    )
    satellite_data = satellite_data.rio.clip(
        site_polygon.to_crs(satellite_data.rio.crs).geometry,
        all_touched=True,
        drop=True,
    ).load()
    satellite_data = harmonize_post_2022(satellite_data)
    if satellite_data.sizes.get("time", 0) == 0:
        print("  No cloud-filtered scenes intersect the site polygon")
        continue

    site_polygon_projected = site_polygon.to_crs(satellite_data.rio.crs)
    site_mask_values = rasterio.features.geometry_mask(
        [geometry.__geo_interface__ for geometry in site_polygon_projected.geometry],
        out_shape=(satellite_data.sizes["y"], satellite_data.sizes["x"]),
        transform=satellite_data.rio.transform(),
        invert=True,
        all_touched=True,
    )
    site_mask = xarray.DataArray(
        site_mask_values,
        coords={"y": satellite_data["y"], "x": satellite_data["x"]},
        dims=("y", "x"),
    )
    site_pixel_count = int(site_mask.sum())
    if site_pixel_count == 0:
        raise ValueError(f"No raster pixels fall inside {site_polygon_file}")

    scl_values = satellite_data["SCL"]
    dry_mask = scl_values.isin([4, 5]) & site_mask
    dry_pixel_counts = dry_mask.sum(dim=["x", "y"]).compute().values
    scl_covered_counts = ((scl_values > 0) & site_mask).sum(dim=["x", "y"]).compute().values

    mask = (site_mask & (scl_values > 0) & ~scl_values.isin(sentinel2.SCL_TO_IGNORE))
    valid_scl_percentage = 100 * mask.sum(dim=["x", "y"]) / site_mask.sum()
    

    green = satellite_data["B03"].astype(numpy.float32)
    swir1 = satellite_data["B11"].astype(numpy.float32)
    denominator = green + swir1
    mndwi_valid = (
        site_mask
        & (scl_values > 0)
        & ~scl_values.isin(sentinel2.SCL_TO_IGNORE)
        & (green > 0)
        & (swir1 > 0)
        & (denominator != 0)
    )
    mndwi = ((green - swir1) / denominator).where(mndwi_valid)
    valid_mndwi_count = mndwi.notnull().sum(dim=["x", "y"]).where(
        lambda count: count > 0
    )
    mndwi_stats = xarray.Dataset(
        {
            "mean_mndwi": mndwi.mean(dim=["x", "y"], skipna=True),
            "median_mndwi": mndwi.median(dim=["x", "y"], skipna=True),
            "std_mndwi": mndwi.std(dim=["x", "y"], skipna=True),
            "fraction_mndwi_gt_0": (mndwi > 0).sum(dim=["x", "y"]) / valid_mndwi_count,
            "fraction_mndwi_gt_-0.1": (
                (mndwi > -0.1).sum(dim=["x", "y"]) / valid_mndwi_count
            ),
        }
    ).compute()

    scene_dates = satellite_data.time.dt.strftime("%Y-%m-%d").values.tolist()
    records = []
    centroid_nzgd49 = site_polygon.to_crs(utils.CRS_NZGD49).geometry.union_all().centroid
    latitude_nzgd49 = float(centroid_nzgd49.y)
    longitude_nzgd49 = float(centroid_nzgd49.x)
    for scene_index, scene_date in enumerate(scene_dates):
        satellite_datetime_utc = pandas.Timestamp(
            satellite_data.time.isel(time=scene_index).values
        )
        satellite_datetime_utc = (
            satellite_datetime_utc.tz_localize("UTC")
            if satellite_datetime_utc.tzinfo is None
            else satellite_datetime_utc.tz_convert("UTC")
        )
        tide_info = get_low_tide_info(
            satellite_datetime_utc, latitude_nzgd49, longitude_nzgd49
        )
        records.append(
            {
                "site": site_name,
                "date": scene_date,
                "satellite_time_utc": satellite_datetime_utc,
                "cloud_cover_pct": cloud_cover_by_date.get(scene_date, numpy.nan),
                **tide_info,
                "dry_area_pct": 100 * int(dry_pixel_counts[scene_index]) / site_pixel_count,
                "scl_coverage_pct": (
                    100 * int(scl_covered_counts[scene_index]) / site_pixel_count
                ),
                "valid_scl_pct": float(valid_scl_percentage[scene_index]),
                **{
                    metric: float(mndwi_stats[metric].isel(time=scene_index).values)
                    for metric in mndwi_stats.data_vars
                },
            }
        )

    site_results[site_name] = pandas.DataFrame.from_records(records).reset_index(drop=True)
    site_results[site_name].to_csv(
        site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_low_tide_dry_area.csv",
        index=False,
    )
    print(f"  Saved {len(site_results[site_name])} scene records to {site_output_path}")

    rgb_cards_by_index = {}
    scl_cards_by_index = {}
    for scene_index, row in site_results[site_name].iterrows():
        rgb = numpy.stack(
            [
                satellite_data[band].isel(time=scene_index).values
                for band in sentinel2.RGB_BANDS
            ],
            axis=-1,
        ).astype(numpy.float32)
        rgb_valid = numpy.isfinite(rgb).all(axis=-1) & (rgb > 0).all(axis=-1)
        rgb_display = numpy.ones_like(rgb)
        if rgb_valid.any():
            for channel in range(3):
                pixels = rgb[..., channel][rgb_valid]
                low, high = numpy.percentile(pixels, [2, 98])
                if high > low:
                    rgb_display[..., channel] = numpy.clip(
                        (rgb[..., channel] - low) / (high - low), 0, 1
                    )
        rgb_display[~rgb_valid] = 1
        low_local = pandas.Timestamp(row["nearest_low_tide_time_utc"]).tz_convert(
            "Pacific/Auckland"
        )
        caption = (
            f"Dry {row['dry_area_pct']:.1f}% | MNDWI {row['mean_mndwi']:.3f} | "
            f"Low {low_local:%d %b %H:%M} ({row['time_from_low_tide']}, "
            f"{row['nearest_low_tide_height_msl']:.2f} m MSL) | Valid SCL % {row['valid_scl_pct']}"
        )
        safe_date = html_module.escape(str(row["date"]))

        rgb_buffer = io.BytesIO()
        rgb_figure, rgb_axis = plt.subplots(figsize=(5, 3.75))
        rgb_axis.imshow(rgb_display)
        rgb_axis.axis("off")
        rgb_figure.savefig(
            rgb_buffer,
            format="jpg",
            dpi=90,
            bbox_inches="tight",
            pad_inches=0,
            pil_kwargs={"quality": 84, "optimize": True},
        )
        plt.close(rgb_figure)
        rgb_base64 = base64.b64encode(rgb_buffer.getvalue()).decode("ascii")
        rgb_cards_by_index[scene_index] = (
            f'<article class="scene"><h2>{safe_date}</h2>'
            f'<img src="data:image/jpeg;base64,{rgb_base64}" '
            f'alt="{site_name} RGB {safe_date}">'
            f'<div class="caption">{caption}</div></article>'
        )

        scl_array = satellite_data["SCL"].isel(time=scene_index).values
        scl_rgb = numpy.full((*scl_array.shape, 3), 238, dtype=numpy.uint8)
        for class_code, _, color in scl_classes:
            rgb_color = tuple(int(color[index:index + 2], 16) for index in (1, 3, 5))
            scl_rgb[(scl_array == class_code) & site_mask_values] = rgb_color
        scl_buffer = io.BytesIO()
        scl_figure, scl_axis = plt.subplots(figsize=(5, 3.75))
        scl_axis.imshow(scl_rgb)
        scl_axis.axis("off")
        scl_figure.savefig(
            scl_buffer,
            format="jpg",
            dpi=90,
            bbox_inches="tight",
            pad_inches=0,
            pil_kwargs={"quality": 90, "optimize": True},
        )
        plt.close(scl_figure)
        scl_base64 = base64.b64encode(scl_buffer.getvalue()).decode("ascii")
        scl_cards_by_index[scene_index] = (
            f'<article class="scene"><h2>{safe_date}</h2>'
            f'<img src="data:image/jpeg;base64,{scl_base64}" '
            f'alt="{site_name} SCL {safe_date}">'
            f'<div class="caption">{caption}</div></article>'
        )

    gallery_specs = [
        ("dry_area_pct", "dry_area"),
        ("mean_mndwi", "mean_mndwi"),
        ("median_mndwi", "median_mndwi"),
    ]
    for sort_column, sort_suffix in gallery_specs:
        ordered_indices = site_results[site_name].sort_values(
            sort_column, ascending=False, kind="stable"
        ).index
        make_gallery(
            site_name,
            f"RGB scenes by {sort_column}",
            [rgb_cards_by_index[index] for index in ordered_indices],
            site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_rgb_by_{sort_suffix}.html",
        )
        make_gallery(
            site_name,
            f"SCL scenes by {sort_column}",
            [scl_cards_by_index[index] for index in ordered_indices],
            site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_scl_by_{sort_suffix}.html",
            scl_legend_html,
        )

    scatter_data = site_results[site_name].copy()
    acquisition_times = pandas.to_datetime(scatter_data["satellite_time_utc"], utc=True)
    scatter_data["date_color"] = acquisition_times.astype("int64") / 1e9
    year_midpoints = acquisition_times.groupby(acquisition_times.dt.year).median()
    color_ticks = (year_midpoints.astype("int64") / 1e9).tolist()
    color_tick_labels = year_midpoints.index.astype(str).tolist()
    scatter_data["abs_hours_from_low_tide"] = scatter_data["hours_from_low_tide"].abs()
    scatter = px.scatter(
        scatter_data,
        x="abs_hours_from_low_tide",
        y="dry_area_pct",
        color="date_color",
        color_continuous_scale="Viridis",
        hover_name="date",
        hover_data={
            "time_from_low_tide": True,
            "nearest_low_tide_height_msl": ":.2f",
            "cloud_cover_pct": ":.1f",
            "valid_scl_pct": ":.1f",
            "scl_coverage_pct": ":.1f",
            "mean_mndwi": ":.3f",
            "date_color": False,
        },
        labels={
            "abs_hours_from_low_tide": "Absolute duration from low tide (hours)",
            "dry_area_pct": "Dry area (SCL 4+5, % of site)",
            "date_color": "Acquisition year",
        },
        title=f"{site_name}: Dry area vs time from low tide ({DATE_RANGE_LABEL})",
    )
    scatter.update_layout(
        height=650,
        hovermode="closest",
        coloraxis_colorbar={
            "title": "Acquisition year",
            "tickvals": color_ticks,
            "ticktext": color_tick_labels,
        },
        yaxis={"range": [0, 100]},
    )
    scatter_file = site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_dry_vs_low_tide.html"
    scatter.write_html(scatter_file, include_plotlyjs=True, auto_open=False)


    scatter = px.scatter(
        scatter_data,
        x="abs_hours_from_low_tide",
        y="mean_mndwi",
        color="date_color",
        color_continuous_scale="Viridis",
        hover_name="date",
        hover_data={
            "time_from_low_tide": True,
            "nearest_low_tide_height_msl": ":.2f",
            "cloud_cover_pct": ":.1f",
            "valid_scl_pct": ":.1f",
            "scl_coverage_pct": ":.1f",
            "mean_mndwi": ":.3f",
            "date_color": False,
        },
        labels={
            "abs_hours_from_low_tide": "Absolute duration from low tide (hours)",
            "dry_area_pct": "Dry area (SCL 4+5, % of site)",
            "date_color": "Acquisition year",
        },
        title=f"{site_name}: Dry area vs time from low tide ({DATE_RANGE_LABEL})",
    )
    scatter.update_layout(
        height=650,
        hovermode="closest",
        coloraxis_colorbar={
            "title": "Acquisition year",
            "tickvals": color_ticks,
            "ticktext": color_tick_labels,
        },
        yaxis={"range": [-.2, 1.2]},
    )
    scatter_file = site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_mean_mndwi_vs_low_tide.html"
    scatter.write_html(scatter_file, include_plotlyjs=True, auto_open=False)

    # plot mean mndwi after clody images removed
    scatter_data = site_results[site_name].copy()
    scatter_data = scatter_data[scatter_data['valid_scl_pct'] > 99]
    acquisition_times = pandas.to_datetime(scatter_data["satellite_time_utc"], utc=True)
    scatter_data["date_color"] = acquisition_times.astype("int64") / 1e9
    year_midpoints = acquisition_times.groupby(acquisition_times.dt.year).median()
    color_ticks = (year_midpoints.astype("int64") / 1e9).tolist()
    color_tick_labels = year_midpoints.index.astype(str).tolist()
    scatter_data["abs_hours_from_low_tide"] = scatter_data["hours_from_low_tide"].abs()

    scatter = px.scatter(
        scatter_data,
        x="abs_hours_from_low_tide",
        y="mean_mndwi",
        color="date_color",
        color_continuous_scale="Viridis",
        hover_name="date",
        hover_data={
            "time_from_low_tide": True,
            "nearest_low_tide_height_msl": ":.2f",
            "cloud_cover_pct": ":.1f",
            "valid_scl_pct": ":.1f",
            "scl_coverage_pct": ":.1f",
            "mean_mndwi": ":.3f",
            "date_color": False,
        },
        labels={
            "abs_hours_from_low_tide": "Absolute duration from low tide (hours)",
            "dry_area_pct": "Dry area (SCL 4+5, % of site)",
            "date_color": "Acquisition year",
        },
        title=f"{site_name}: Dry area vs time from low tide ({DATE_RANGE_LABEL})",
    )
    scatter.update_layout(
        height=650,
        hovermode="closest",
        coloraxis_colorbar={
            "title": "Acquisition year",
            "tickvals": color_ticks,
            "ticktext": color_tick_labels,
        },
        yaxis={"range": [-.2, 1.2]},
    )
    scatter_file = site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_low_cloud_mean_mndwi_vs_low_tide.html"
    scatter.write_html(scatter_file, include_plotlyjs=True, auto_open=False)
    

    print(f"  Saved scatter: {scatter_file}")
    display(site_results[site_name])
    for sort_suffix in ("dry_area", "mean_mndwi"):
        display(
            FileLink(
                str(site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_rgb_by_{sort_suffix}.html")
            )
        )
        display(
            FileLink(
                str(site_output_path / f"{site_name}_{DATE_RANGE_LABEL}_scl_by_{sort_suffix}.html")
            )
        )
    display(FileLink(str(scatter_file)))


Ihutai: search 2016-2025 Sentinel-2 scenes
  2016: 53 STAC items
  2017: 44 STAC items
  2018: 73 STAC items
  2019: 68 STAC items
  2020: 72 STAC items
  2021: 74 STAC items
  2022: 115 STAC items
  2023: 129 STAC items
  2024: 74 STAC items
  2025: 70 STAC items
  STAC items across range: 772; under cloud cutoff: 110
  Saved 97 scene records to C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai
  Saved C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_rgb_by_dry_area.html
  Saved C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_scl_by_dry_area.html
  Saved C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_rgb_by_mean_mndwi.html
  Saved C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_scl_by_mean_mndwi.html
  Saved C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_rgb_by_median_mndwi.html
  Saved C:\Local\repos

,site,date,satellite_time_utc,cloud_cover_pct,nearest_low_tide_height_msl,nearest_low_tide_time_utc,time_from_low_tide,hours_from_low_tide,dry_area_pct,scl_coverage_pct,valid_scl_pct,mean_mndwi,median_mndwi,std_mndwi,fraction_mndwi_gt_0,fraction_mndwi_gt_-0.1
0,Ihutai,2016-02-09,2016-02-09 22:25:32.030000+00:00,1.493319,-1.11,2016-02-09 23:48:00+00:00,1h 22m after,1.374436,6.976334,99.990604,99.682894,0.407335,0.642369,0.410964,0.776483,0.831800
1,Ihutai,2016-02-19,2016-02-19 22:25:32.029000+00:00,2.657386,-0.88,2016-02-19 20:41:00+00:00,1h 45m after,1.742230,8.410359,99.971813,99.255388,0.242533,0.140777,0.454539,0.641680,0.751594
2,Ihutai,2016-04-09,2016-04-09 22:25:32.030000+00:00,9.514942,-1.23,2016-04-10 00:47:00+00:00,2h 21m after,2.357769,7.820776,100.000000,68.379823,0.430036,0.527711,0.443890,0.792261,0.864158
3,Ihutai,2016-04-26,2016-04-26 22:15:52.028000+00:00,0.008174,-0.71,2016-04-27 02:09:00+00:00,3h 53m after,3.885548,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
4,Ihutai,2016-05-06,2016-05-06 22:15:52.027000+00:00,0.110175,-1.25,2016-05-06 22:34:00+00:00,0h 18m after,0.302215,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,Ihutai,2025-09-14,2025-09-14 22:25:39.024000+00:00,7.040510,-0.94,2025-09-14 16:19:00+00:00,6h 07m after,6.110840,12.769981,100.000000,100.000000,0.710656,0.937239,0.450412,0.867469,0.884857
93,Ihutai,2025-09-29,2025-09-29 22:26:01.025000+00:00,6.435211,-0.60,2025-09-29 16:08:00+00:00,6h 18m after,6.300285,14.317928,100.000000,100.000000,0.651604,0.878646,0.453175,0.851389,0.868091
94,Ihutai,2025-10-24,2025-10-24 22:25:39.024000+00:00,0.326880,-0.67,2025-10-25 01:14:00+00:00,2h 48m after,2.805827,12.543308,100.000000,100.000000,0.685268,0.920768,0.449063,0.862403,0.887056
95,Ihutai,2025-10-29,2025-10-29 22:26:01.025000+00:00,2.153105,-0.61,2025-10-29 16:15:00+00:00,6h 11m after,6.183618,14.255681,100.000000,100.000000,0.643408,0.863226,0.446076,0.845945,0.862998


C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_rgb_by_dry_area.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_scl_by_dry_area.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites\Ihutai\Ihutai_2016-2025_rgb_by_mean_mndwi.html

...

C:\Local\repos\seagrass-detection\data\analysis\all_sites\IveyBay_ThePoint_LeftBank_Oct24\IveyBay_ThePoint_LeftBank_Oct24_2016-2025_rgb_by_dry_area.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites\IveyBay_ThePoint_LeftBank_Oct24\IveyBay_ThePoint_LeftBank_Oct24_2016-2025_scl_by_dry_area.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites\IveyBay_ThePoint_LeftBank_Oct24\IveyBay_ThePoint_LeftBank_Oct24_2016-2025_rgb_by_mean_mndwi.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites\IveyBay_ThePoint_LeftBank_Oct24\IveyBay_ThePoint_LeftBank_Oct24_2016-2025_scl_by_mean_mndwi.html

C:\Local\repos\seagrass-detection\data\analysis\all_sites\IveyBay_ThePoint_LeftBank_Oct24\IveyBay_ThePoint_LeftBank_Oct24_2016-2025_low_cloud_mean_mndwi_vs_low_tide.html